2. Projektni Zadatak: Razvoj Klasifikatora za Analizu
Sentimenta u PySpark-u

In [79]:
from pyspark.sql import SparkSession

sc = SparkSession.builder.appName("FinancialSentimentAnalysis").getOrCreate()


In [80]:
data = sc.read.csv("data.csv", header=True, inferSchema=True, quote='"', escape='"')
data.show(truncate=80)

+--------------------------------------------------------------------------------+---------+
|                                                                        Sentence|Sentiment|
+--------------------------------------------------------------------------------+---------+
|The GeoSolutions technology will leverage Benefon 's GPS solutions by providi...| positive|
|                         $ESI on lows, down $1.50 to $2.50 BK a real possibility| negative|
|For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m fro...| positive|
|According to the Finnish-Russian Chamber of Commerce , all the major construc...|  neutral|
|The Swedish buyout firm has sold its remaining 22.4 percent stake , almost ei...|  neutral|
|                                 $SPY wouldn't be surprised to see a green close| positive|
|                        Shell's $70 Billion BG Deal Meets Shareholder Skepticism| negative|
|SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANGE RELEASE OCTOBER 14 , 

In [81]:
data.printSchema()

root
 |-- Sentence: string (nullable = true)
 |-- Sentiment: string (nullable = true)



In [82]:
from pyspark.sql.functions import count, when, col

data.select([count(when(col(c).isNull(), c)).alias(c) for c in data.columns]).show()

+--------+---------+
|Sentence|Sentiment|
+--------+---------+
|       0|        0|
+--------+---------+



In [83]:
data.groupBy("sentiment").count().show()

+---------+-----+
|sentiment|count|
+---------+-----+
| positive| 1852|
|  neutral| 3130|
| negative|  860|
+---------+-----+



In [84]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="Sentiment",
    outputCol="label"
)

data = label_indexer.fit(data).transform(data)
data.select("Sentiment", "label").show(10)

+---------+-----+
|Sentiment|label|
+---------+-----+
| positive|  1.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
|  neutral|  0.0|
| positive|  1.0|
| negative|  2.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
+---------+-----+
only showing top 10 rows


In [85]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_data.count())
print("Test:", test_data.count())

Train: 4725
Test: 1117


In [86]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="Sentence",
    outputCol="tokens"
)

print("Tokens:")
tokenizer.transform(train_data).select("Sentence", "tokens").show(10, truncate=60)


Tokens:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                    Sentence|                                                      tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|#Apple breaks major support, here are some levels to watc...|[#apple, breaks, major, support,, here, are, some, levels...|
|#Apple up almost 20% from its February lows with very lit...|[#apple, up, almost, 20%, from, its, february, lows, with...|
|#FusionIQ NEW Positive Timing Signal on $SBUX Today https...|[#fusioniq, new, positive, timing, signal, on, $sbux, tod...|
|#LongPos $TSLA 256 Break-out thru 50 & 200- DMA (197-230)...|[#longpos, $tsla, 256, break-out, thru, 50, &, 200-, dma,...|
|#NDX component Tesla has announced it is recalling 2700 M...|[#ndx, component, tesla, has, announced, it, is, recallin...|


In [87]:
from pyspark.ml.feature import StopWordsRemover

stop_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

print("Filtered Tokens:")
stop_remover.transform(tokenizer.transform(train_data)).select("Sentence", "filtered_tokens").show(10, truncate=60)

Filtered Tokens:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                    Sentence|                                             filtered_tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|#Apple breaks major support, here are some levels to watc...|[#apple, breaks, major, support,, levels, watch, -, http:...|
|#Apple up almost 20% from its February lows with very lit...|[#apple, almost, 20%, february, lows, little, fanfare., $...|
|#FusionIQ NEW Positive Timing Signal on $SBUX Today https...|[#fusioniq, new, positive, timing, signal, $sbux, today, ...|
|#LongPos $TSLA 256 Break-out thru 50 & 200- DMA (197-230)...|[#longpos, $tsla, 256, break-out, thru, 50, &, 200-, dma,...|
|#NDX component Tesla has announced it is recalling 2700 M...|[#ndx, component, tesla, announced, recalling, 2700, 

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

# To avoid PySpark Python worker Serialization errors or BrokenPipeErrors, 
# it is safer to instantiate objects like PorterStemmer inside the UDF or 
# mapPartitions. For a standard UDF, doing it concisely helps:
def stem_tokens(tokens):
    if not tokens:
        return []
    import nltk
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
    return [stemmer.stem(token) for token in tokens]

stem_udf = udf(stem_tokens, ArrayType(StringType()))

# Register the UDF with the SparkSession so it can be used in an SQLTransformer pipeline later
sc.udf.register("stem_udf", stem_tokens, ArrayType(StringType()))

# Apply the stemming UDF to the filtered tokens
stemmed_data = stop_remover.transform(tokenizer.transform(train_data)).withColumn("stemmed_tokens", stem_udf(col("filtered_tokens")))
print("Stemmed Tokens:")
stemmed_data.select("Sentence", "stemmed_tokens").show(10, truncate=60)


Stemmed Tokens:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                    Sentence|                                              stemmed_tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|#Apple breaks major support, here are some levels to watc...|[#appl, break, major, support,, level, watch, -, http://s...|
|#Apple up almost 20% from its February lows with very lit...| [#appl, almost, 20%, februari, low, littl, fanfare., $aapl]|
|#FusionIQ NEW Positive Timing Signal on $SBUX Today https...|[#fusioniq, new, posit, time, signal, $sbux, today, https...|
|#LongPos $TSLA 256 Break-out thru 50 & 200- DMA (197-230)...|[#longpo, $tsla, 256, break-out, thru, 50, &, 200-, dma, ...|
|#NDX component Tesla has announced it is recalling 2700 M...|[#ndx, compon, tesla, announc, recal, 2700, model, x, 

26/05/23 02:42:23 WARN SimpleFunctionRegistry: The function stem_udf replaced a previously registered function.


In [ ]:
import nltk
from pyspark.sql.functions import col

# Check if 'wordnet' corpus is already downloaded, if not, download it just once
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    print("Downloading 'wordnet' corpus...")
    nltk.download('wordnet', quiet=True)

def lemmatize_tokens(tokens):
    if not tokens:
        return []
    import nltk
    from nltk.stem import WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in tokens]

lemmatize_udf = udf(lemmatize_tokens, ArrayType(StringType()))

# Register the UDF with the SparkSession so it can be used in an SQLTransformer pipeline later
sc.udf.register("lemmatize_udf", lemmatize_tokens, ArrayType(StringType()))

# Apply the lemmatization UDF to the filtered tokens
lemmatized_data = stop_remover.transform(tokenizer.transform(train_data)).withColumn("lemmatized_tokens", lemmatize_udf(col("filtered_tokens")))

print("Lemmatized Tokens:")
lemmatized_data.select("Sentence", "lemmatized_tokens").show(10, truncate=60)


Lemmatized Tokens:


26/05/23 03:02:26 WARN SimpleFunctionRegistry: The function lemmatize_udf replaced a previously registered function.


+------------------------------------------------------------+------------------------------------------------------------+
|                                                    Sentence|                                           lemmatized_tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|#Apple breaks major support, here are some levels to watc...|[#apple, break, major, support,, level, watch, -, http://...|
|#Apple up almost 20% from its February lows with very lit...|[#apple, almost, 20%, february, low, little, fanfare., $a...|
|#FusionIQ NEW Positive Timing Signal on $SBUX Today https...|[#fusioniq, new, positive, timing, signal, $sbux, today, ...|
|#LongPos $TSLA 256 Break-out thru 50 & 200- DMA (197-230)...|[#longpos, $tsla, 256, break-out, thru, 50, &, 200-, dma,...|
|#NDX component Tesla has announced it is recalling 2700 M...|[#ndx, component, tesla, announced, recalling, 2700, mode...|
|#Tesla:

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, CountVectorizer, IDF, Word2Vec, NGram
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Base stages
text_processing_transformer = SQLTransformer(
    statement="SELECT *, stem_udf(filtered_tokens) AS stemmed_tokens, lemmatize_udf(filtered_tokens) AS lemmatized_tokens FROM __THIS__"
)
base_stages = [tokenizer, stop_remover, text_processing_transformer]

# 2. Define Feature Extraction variants for both sets of tokens
feature_variants = {}
for text_col in ["stemmed_tokens", "lemmatized_tokens"]:
    feature_variants[f"tfidf_{text_col}"] = [
        CountVectorizer(inputCol=text_col, outputCol="raw_features", vocabSize=5000, minDF=5.0),
        IDF(inputCol="raw_features", outputCol="features")
    ]
    feature_variants[f"bigram_tfidf_{text_col}"] = [
        NGram(n=2, inputCol=text_col, outputCol="bigrams"),
        CountVectorizer(inputCol="bigrams", outputCol="raw_features", vocabSize=5000, minDF=5.0),
        IDF(inputCol="raw_features", outputCol="features")
    ]
    feature_variants[f"word2vec_{text_col}"] = [
        Word2Vec(vectorSize=100, minCount=1, inputCol=text_col, outputCol="features")
    ]

# 3. Define Models and Hyperparameter Grids
lr = LogisticRegression(featuresCol="features", labelCol="label")
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)

grids = {
    "lr": (lr, ParamGridBuilder() \
        .addGrid(lr.maxIter, [100]) \
        .addGrid(lr.regParam, [0.01, 0.1]) \
        .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
        .build()),
    "dt": (dt, ParamGridBuilder() \
        .addGrid(dt.maxDepth, [8, 10]) \
        .addGrid(dt.minInstancesPerNode, [2, 5]) \
        .build()),
    "rf": (rf, ParamGridBuilder() \
        .addGrid(rf.numTrees, [100]) \
        .addGrid(rf.maxDepth, [10]) \
        .addGrid(rf.featureSubsetStrategy, ["sqrt"]) \
        .build())
}

# 4. Evaluators
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# 5. Run experiments with Pipelines and CrossValidator
results_table = []
best_overall_model = None
best_f1 = 0.0

for feat_name, feat_stages in feature_variants.items():
    for model_name, (model, grid) in grids.items():
        print(f"Running pipeline + CrossValidator for: {feat_name} + {model_name}")
        
        # Build the full pipeline
        pipeline = Pipeline(stages=base_stages + feat_stages + [model])
        
        # Using CrossValidator. A practical alternative for big data is TrainValidationSplit
        cv = CrossValidator(estimator=pipeline,
                            estimatorParamMaps=grid,
                            evaluator=evaluator_f1,
                            numFolds=2,    # Reduced folds for demonstration speed
                            seed=42)
        
        # Fit on train_data
        cv_model = cv.fit(train_data)
        
        # Best model results on test_data
        predictions = cv_model.transform(test_data)
        f1_score = evaluator_f1.evaluate(predictions)
        accuracy = evaluator_acc.evaluate(predictions)
        
        print(f"  -> Best params yielded F1: {f1_score:.4f}, Accuracy: {accuracy:.4f}\n")
        
        results_table.append({
            "experiment": f"{feat_name}_{model_name}",
            "f1": f1_score,
            "accuracy": accuracy,
            "best_model": cv_model.bestModel
        })
        
        if f1_score > best_f1:
            best_f1 = f1_score
            best_overall_model = cv_model.bestModel


Running pipeline + CrossValidator for: tfidf_stemmed_tokens + lr
  -> Best params yielded F1: 0.7102, Accuracy: 0.7296

Running pipeline + CrossValidator for: tfidf_stemmed_tokens + dt


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe
Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe
Traceback (most recent

  -> Best params yielded F1: 0.6630, Accuracy: 0.7117

Running pipeline + CrossValidator for: tfidf_stemmed_tokens + rf


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe
26/05/23 02:42:57 WARN DAGScheduler: Broadcasting large task binary with size 1037.9 KiB
26/05/23 02:42:57 WARN DAGScheduler: Broadcasting large task binary with size 1222.5 KiB
Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib

  -> Best params yielded F1: 0.5605, Accuracy: 0.6491

Running pipeline + CrossValidator for: bigram_tfidf_stemmed_tokens + lr
  -> Best params yielded F1: 0.5346, Accuracy: 0.6007

Running pipeline + CrossValidator for: bigram_tfidf_stemmed_tokens + dt


...

  -> Best params yielded F1: 0.4636, Accuracy: 0.5936

Running pipeline + CrossValidator for: word2vec_lemmatized_tokens + lr
  -> Best params yielded F1: 0.5729, Accuracy: 0.6195

Running pipeline + CrossValidator for: word2vec_lemmatized_tokens + dt


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe
Traceback (most recent call last):                                              
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError:

  -> Best params yielded F1: 0.5854, Accuracy: 0.6061

Running pipeline + CrossValidator for: word2vec_lemmatized_tokens + rf


Traceback (most recent call last):
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vesco/Documents/Projects/big-data-ml/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe
26/05/23 02:46:21 WARN DAGScheduler: Broadcasting large task binary with size 1044.9 KiB
26/05/23 02:46:22 WARN DAGScheduler: Broadcasting large task binary with size 1694.2 KiB
26/05/23 02:46:22 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/05/23 02:46:22 WARN DAGScheduler: Broadcasting large task binary with size 3.6 MiB
26/05/23 02:46:23 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
26/05/23 02:46:23 WARN DAGScheduler: Broadcasting large task binary with si

  -> Best params yielded F1: 0.5942, Accuracy: 0.6312



In [91]:
# Output the best overall model
best_experiment = max(results_table, key=lambda x: x["f1"])

print("Best configuration:")
print("Experiment:", best_experiment["experiment"])
print(f"F1 Score: {best_experiment['f1']:.4f}")
print(f"Accuracy: {best_experiment['accuracy']:.4f}")
print("Pipeline Details:")
print(best_experiment["best_model"].stages)


Best configuration:
Experiment: tfidf_stemmed_tokens_lr
F1 Score: 0.7102
Accuracy: 0.7296
Pipeline Details:
[Tokenizer_6693390214ac, StopWordsRemover_070edf5c40f0, SQLTransformer_fcb554c4c684, CountVectorizerModel: uid=CountVectorizer_8323c99880b6, vocabularySize=1847, IDFModel: uid=IDF_af53fdcdbb30, numDocs=4725, numFeatures=1847, LogisticRegressionModel: uid=LogisticRegression_51e1eef536fd, numClasses=3, numFeatures=1847]


In [92]:
import pandas as pd

# Creating a clean DataFrame containing the results
clean_results = [
    {"experiment": r["experiment"], "f1": r["f1"], "accuracy": r["accuracy"]} 
    for r in results_table
]

results_df = pd.DataFrame(clean_results)
results_df = results_df.sort_values(by="f1", ascending=False)
results_df


,experiment,f1,accuracy
0,tfidf_stemmed_tokens_lr,0.710217,0.729633
9,tfidf_lemmatized_tokens_lr,0.709812,0.730528
1,tfidf_stemmed_tokens_dt,0.663048,0.711728
10,tfidf_lemmatized_tokens_dt,0.653028,0.703671
6,word2vec_stemmed_tokens_lr,0.595634,0.637422
17,word2vec_lemmatized_tokens_rf,0.594181,0.631155
16,word2vec_lemmatized_tokens_dt,0.585381,0.606088
8,word2vec_stemmed_tokens_rf,0.585378,0.624888
7,word2vec_stemmed_tokens_dt,0.575136,0.617726
15,word2vec_lemmatized_tokens_lr,0.572901,0.619517


Interpretacija rezultata

Nakon sprovedene analize koja je direktno uporedila _Stemming_ i _Lemmatization_ pristupe preprocesiranja teksta uz razlicite feature i model varijante, moze se zakljuciti sledece:

Najbolji rezultat na Financial Sentiment Analysis datasetu ostvario je **Logistic Regression sa TF-IDF reprezentacijom**. 
Iako su metode preprocesiranja imale vrlo slicne rezultate, varijanta sa **Stemming** tehnikom (tfidf_stemmed_tokens_lr) dala je za nijansu bolji F1 Score (≈ 0.710), dok je **Lemmatization** (tfidf_lemmatized_tokens_lr) ostvarila blago visi tacan pogodak prepoznavanja klasa, odnosno Accuracy (≈ 0.730). 

Kljucna zapazanja:
1. **Preprocesiranje (Stemming vs Lemmatization):** Obe tehnike proizvode prakticno isti nivo performansi. U ovom konkretnom slucaju, agresivnije skracivanje reci (Stemming) jednako efikasno grupise osnovno znacenje sentimenta kao i sofisticiranija lematizacija.
2. **Ekstrakcija Featuresa:** TF-IDF je dramaticno uspesniji od prostog proredjenog prebrojavanja. Pokazao se znatno boljim od Word2Vec (koji zahteva znatno vecu bazu reci da bi dostigao solidne vrednosti) i ekstremno boljim od bigrama, potvrdjujuci da su kljucne pojedinacne finansijske reci (unigrami) najjaci indikatori sentimenta.
3. **Modeli Klasifikatori:** Linearni Logistic Regression dominira. Random Forest i Decision Tree pate u visokodimenzionalnim vektorskim prostorima koje stvaraju tekstualni podaci i imaju drasticno gori F1 skor (≈ 0.65 za DT i ≈ 0.56 za RF sa TF-IDF)

Zakljucak

Pobednicki pipeline pristup je **TF-IDF + Logistic Regression**. Zbog manjeg utroska racunarskih resursa pri obradi, moze se preferirati Stemming u odnosu na Lemmatization jer zahteva jednostavnije zavisnosti uz prakticno identican finalni rezultat, stabilnost i tacnost koji leze u cinjenici da se obrasci u pozitivnim i negativnim vestima ponasaju gotovo linearno separabilno u ogromnom text frequency prostoru.